# Elendril Hitchhiker -- Kaggle T4x2 Source-Pinned Experiment

- Exact pinned commit: `c4d34dda5eebd75267995ff86e5fd45f1c80805a`
- Phases supported:
  - `stage0`: Historical `V7.17` `S20-low-lr` frozen checkpoint audit (`--stage0-checkpoint-root`, if mounted).
  - `stage1`: 15-run `S3` dev screening (`B0, A, B, AB, C` x seeds `7, 19, 41`), locking dev `selection_receipt.json`.
  - `stage2`: Conditional follow-ups (`AB+C`, `C+penalty`, `D-external`, `S20-contrast`).
  - `stage3`: Locked-winner `S8`/`S20` scale & formation confirmation on independent `confirmation` + structural holdouts.
- Output directory: `/kaggle/working/elendril_hitchhiker_run`
- Verified deliverable bundles:
  - `exports/ELENDRIL_HITCHHIKER_ANALYSIS.zip`
  - `exports/ELENDRIL_HITCHHIKER_FULL.zip`
  - `exports/FINAL_STATUS.json`

Archive integrity proves byte fidelity, not scientific capability. Only `gate.status == 'FORMATION_QUALIFIED'` on `confirmation` qualifies formation.

In [ ]:
import json, os, shutil, subprocess, tarfile
from pathlib import Path

OWNER_ENABLED = True
PHASE = "stage1"  # 'stage0', 'stage1', 'stage2', or 'stage3'
MINUTES_BUDGET = 540
SOURCE_COMMIT = "c4d34dda5eebd75267995ff86e5fd45f1c80805a"
REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"

STAGE0_CHECKPOINT_ROOT = None  # e.g. Path("/kaggle/input/elendril-v717-s20-checkpoints/checkpoints")
RESUME_ARCHIVE = None          # e.g. Path("/kaggle/input/elendril-hitchhiker-stage1/ELENDRIL_HITCHHIKER_FULL.zip")
WINNING_ARM = None             # required when running PHASE='stage3' directly from a restored Stage 1 bundle
ADMITTED_FOLLOWUPS = []        # optional subset for PHASE='stage2', e.g. ["AB+C", "C+penalty"]

WORK_ROOT = Path("/kaggle/working/elendril_hitchhiker_run")
CODE_ROOT = Path("/kaggle/working/anra_src")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)
CODE_ROOT.mkdir(parents=True, exist_ok=True)

tar_path = Path("/kaggle/working/pinned_repo.tar")
subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CODE_ROOT / ".git_mirror")], check=True)
subprocess.run(["git", "-C", str(CODE_ROOT / ".git_mirror"), "fetch", "--all", "--tags"], check=True)
resolved = subprocess.check_output(["git", "-C", str(CODE_ROOT / ".git_mirror"), "rev-parse", SOURCE_COMMIT], text=True).strip()
assert resolved == SOURCE_COMMIT, f"commit mismatch: {resolved} != {SOURCE_COMMIT}"
subprocess.run(["git", "-C", str(CODE_ROOT / ".git_mirror"), "archive", "--format=tar", f"--output={tar_path}", SOURCE_COMMIT], check=True)
with tarfile.open(tar_path) as tf:
    tf.extractall(CODE_ROOT)
os.chdir(CODE_ROOT)
print("Verified clean source extract at commit", SOURCE_COMMIT)


In [ ]:
cmd = [
    "python", "-m", "tools.elendril_hitchhiker_campaign",
    "--output", str(WORK_ROOT),
    "--source", SOURCE_COMMIT,
    "--phase", PHASE,
    "--minutes", str(MINUTES_BUDGET),
]
if OWNER_ENABLED:
    cmd.append("--owner-enabled")
if STAGE0_CHECKPOINT_ROOT is not None:
    cmd.extend(["--stage0-checkpoint-root", str(STAGE0_CHECKPOINT_ROOT)])
if RESUME_ARCHIVE is not None:
    cmd.extend(["--resume-archive", str(RESUME_ARCHIVE)])
if WINNING_ARM is not None:
    cmd.extend(["--winning-arm", str(WINNING_ARM)])
if ADMITTED_FOLLOWUPS:
    cmd.extend(["--admitted-followups", *ADMITTED_FOLLOWUPS])

print("Launching:", " ".join(cmd))
subprocess.run(cmd, cwd=CODE_ROOT, check=True)
final_status = json.loads((WORK_ROOT / "exports" / "FINAL_STATUS.json").read_text())
print(json.dumps({
    "status": final_status["status"],
    "claim_status": final_status.get("claim_status"),
    "phase": final_status.get("phase"),
    "replay": final_status.get("independent_replay", {}).get("status"),
    "errors": final_status.get("errors"),
}, indent=2))


In [ ]:
from IPython.display import FileLink, display

for name in ("ELENDRIL_HITCHHIKER_ANALYSIS.zip", "ELENDRIL_HITCHHIKER_FULL.zip", "FINAL_STATUS.json"):
    target = WORK_ROOT / "exports" / name
    if target.exists():
        display(FileLink(str(target)))
